# 逻辑回归数学基础

这份 Notebook 用来整理进入逻辑回归之前最需要掌握的数学基础，重点放在：

- 概率与条件概率
- 伯努利分布与二分类标签
- odds 与 log-odds
- sigmoid 函数
- 从线性分数到分类概率
- 逻辑回归中的基本记号

目标不是一开始就把所有推导做完，而是先搭好一条你后面学习逻辑回归时会一直用到的数学主线。

## 1. 为什么要先补这部分数学基础

在线性回归中，我们预测的是一个连续值：

$$
\hat{y} = Xw + b
$$

但在逻辑回归中，我们通常面对的是二分类问题，例如：

- 这封邮件是不是垃圾邮件
- 这个病人是否患病
- 这名学生能否通过考试

这时标签不再是任意实数，而通常写成：

$$
y \in \{0, 1\}
$$

于是问题就变成了：给定特征 $x$，我们如何预测样本属于 1 类的概率？

也就是：

$$
P(y = 1 \mid x)
$$

所以逻辑回归真正的数学入口不是先看代码，而是先理解概率、条件概率、概率分布，以及概率是怎样和线性模型连起来的。

## 2. 基本记号

为了和你前面的线性回归笔记保持一致，我们仍然使用下面这些记号：

$$
X \in \mathbb{R}^{m \times n}
$$

表示特征矩阵，其中：

- $m$ 是样本数
- $n$ 是特征数

第 $i$ 个样本记作：

$$
x^{(i)} \in \mathbb{R}^{n}
$$

标签向量记作：

$$
y \in \{0,1\}^{m}
$$

也就是说，在逻辑回归里，标签向量中的每个元素都只有两种可能：0 或 1。

参数仍然记作：

$$
w \in \mathbb{R}^{n}, \qquad b \in \mathbb{R}
$$

线性部分仍然是：

$$
z = w^T x + b
$$

后面逻辑回归的大部分新内容，都是围绕这个线性分数 $z$ 如何变成概率来展开的。

## 3. 概率的最基本含义

概率可以粗略理解成一件事情发生的可能性大小。

如果事件 $A$ 发生的概率记作：

$$
P(A)
$$

那么它满足：

$$
0 \le P(A) \le 1
$$

其中：

- $P(A)=0$ 表示几乎不可能发生
- $P(A)=1$ 表示一定发生
- $P(A)=0.5$ 表示发生与不发生同样可能

在逻辑回归里，我们最关心的概率通常不是任意事件的概率，而是“样本属于正类的概率”。

例如：

$$
P(y=1 \mid x)
$$

它表示：在样本特征为 $x$ 的条件下，这个样本属于 1 类的概率。

## 4. 条件概率

条件概率是逻辑回归里最核心的概率概念之一。

它的形式是：

$$
P(A \mid B)
$$

意思是：在事件 $B$ 已经发生的条件下，事件 $A$ 发生的概率。

在机器学习里，通常可以这样理解：

- $B$ 是“给定输入特征 $x$”
- $A$ 是“标签属于某一类”

所以逻辑回归学习的不是：

$$
P(y=1)
$$

而是：

$$
P(y=1 \mid x)
$$

这里的差别很重要：

- $P(y=1)$ 只看整体正类比例
- $P(y=1 \mid x)$ 会根据样本自己的特征来改变

机器学习模型真正想做的，就是输入不同的 $x$，输出不同的条件概率。

## 5. 随机变量与伯努利分布

逻辑回归里的标签只有两种取值：0 和 1。

这种只有两种结果的随机变量，很适合用伯努利分布来描述。

如果随机变量 $Y$ 满足：

$$
Y \in \{0,1\}
$$

并且：

$$
P(Y=1)=p, \qquad P(Y=0)=1-p
$$

那么就说 $Y$ 服从参数为 $p$ 的伯努利分布。

记作：

$$
Y \sim \mathrm{Bernoulli}(p)
$$

这和逻辑回归非常匹配，因为逻辑回归的输出正是一个概率 $p$，它表示：

$$
p = P(Y=1 \mid x)
$$

于是我们也就自动得到：

$$
P(Y=0 \mid x)=1-p
$$

这就是二分类概率建模最基础的一层。

### 5.1 正类与负类

在二分类里，我们通常把两个类别写成：

$$
y \in \{0,1\}
$$

其中通常约定：

- $y=1$ 表示正类
- $y=0$ 表示负类

这里的“正”和“负”并不是“好”和“坏”的意思，而只是两个类别的名字。

例如：

- 在垃圾邮件检测里，可以把“是垃圾邮件”记成 $y=1$
- 在疾病预测里，可以把“患病”记成 $y=1$
- 在考试预测里，可以把“通过”记成 $y=1$

也就是说，正类通常就是我们特别关心、并编码为 $1$ 的那一类。

因此逻辑回归最常预测的是：

$$
P(y=1 \mid x)
$$

也就是“样本属于正类的概率”。而属于负类的概率自然就是：

$$
P(y=0 \mid x)=1-P(y=1 \mid x)
$$


## 6. 伯努利分布为什么能写成一个统一公式

对于单个样本，如果它的真实标签 $y$ 只能取 0 或 1，那么：

- 当 $y=1$ 时，我们希望对应的概率是 $p$
- 当 $y=0$ 时，我们希望对应的概率是 $1-p$

这两种情况可以统一写成：

$$
P(Y=y \mid x) = p^y (1-p)^{1-y}
$$

你可以自己代入验证：

当 $y=1$ 时：

$$
p^1(1-p)^0 = p
$$

当 $y=0$ 时：

$$
p^0(1-p)^1 = 1-p
$$

这个统一公式后面会直接连到逻辑回归的似然函数与交叉熵损失，所以值得先记住。

## 7. odds 与 log-odds

除了直接讨论概率 $p$，逻辑回归里还有两个特别重要的量：odds 和 log-odds。

### 7.1 odds

如果一个事件发生的概率是 $p$，不发生的概率是 $1-p$，那么它的 odds 定义为：

$$
\mathrm{odds} = \frac{p}{1-p}
$$

它表示“发生”和“不发生”的相对比值。

例如：

- 如果 $p=0.5$，那么 odds = 1
- 如果 $p=0.8$，那么 odds = 4
- 如果 $p=0.2$，那么 odds = 0.25

### 7.2 log-odds

再对 odds 取对数，就得到 log-odds：

$$
\log \frac{p}{1-p}
$$

它的重要性在于：

- 概率 $p$ 被限制在 $(0,1)$
- odds 被限制在 $(0,+\infty)$
- log-odds 则可以取任意实数

而线性模型 $w^T x+b$ 恰好也是任意实数。

于是，逻辑回归的核心思想之一就是：

$$
\log \frac{p}{1-p} = w^T x + b
$$

这句话非常关键，因为它把“概率”和“线性模型”真正连接起来了。

### 7.3 为什么要引入 log-odds

我们引入 log-odds，核心是为了把“概率问题”变成“线性模型能处理的问题”。

需要注意下面三个量的取值范围：

- 概率 $p$ 的范围是 $0 < p < 1$
- odds 的范围是 $0 < \dfrac{p}{1-p} < +\infty$
- log-odds 的范围是 $-\infty < \log\dfrac{p}{1-p} < +\infty$

而线性模型

$$
w^T x + b
$$

本身可以取任意实数值，所以它天然适合去建模一个定义在整个实数轴上的量。

如果我们直接写：

$$
p = w^T x + b
$$

就会出问题，因为右边可能小于 $0$ 或大于 $1$，这不符合概率的定义。

如果直接建模 odds：

$$
\frac{p}{1-p}
$$

它虽然总是正数，但仍然不如整个实数轴上的量方便。

于是最自然的选择就是对 odds 再取对数，得到：

$$
\log\frac{p}{1-p}
$$

这样它就能和线性模型无缝对接：

$$
\log\frac{p}{1-p} = w^T x + b
$$

还有一个很直观的点：

- 当 $p=0.5$ 时，有 $\log\dfrac{p}{1-p}=0$
- 当 $p>0.5$ 时，有 $\log\dfrac{p}{1-p}>0$
- 当 $p<0.5$ 时，有 $\log\dfrac{p}{1-p}<0$

所以 $0$ 会自然地成为分类边界。

## 8. 从 log-odds 推出 sigmoid

如果我们设：

$$
z = w^T x + b
$$

并令：

$$
\log \frac{p}{1-p} = z
$$

那么就可以一步步解出 $p$。

先对两边取指数：

$$
\frac{p}{1-p} = e^z
$$

然后整理：

$$
p = e^z(1-p)
$$

$$
p = e^z - e^z p
$$

$$
p(1+e^z)=e^z
$$

$$
p = \frac{e^z}{1+e^z} = \frac{1}{1+e^{-z}}
$$

这就是 sigmoid 函数：

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

也就是说，逻辑回归不是随便选了一个函数，而是从 log-odds 的线性假设自然推出了 sigmoid。

## 9. sigmoid 函数的性质

sigmoid 函数：

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

有几个特别重要的性质：

### 9.1 值域在 0 到 1 之间

$$
0 < \sigma(z) < 1
$$

这正好适合拿来表示概率。

### 9.2 当 $z=0$ 时，输出为 0.5

$$
\sigma(0)=0.5
$$

这意味着线性分数为 0 时，模型认为两类一样可能。

### 9.3 当 $z$ 越大时，输出越接近 1

如果 $z \to +\infty$，那么：

$$
\sigma(z) \to 1
$$

### 9.4 当 $z$ 越小时，输出越接近 0

如果 $z \to -\infty$，那么：

$$
\sigma(z) \to 0
$$

### 9.5 sigmoid 是单调递增的

这表示：

- 线性分数更大
- 属于正类的概率也更大

这让模型的解释很自然。

## 10. 从线性分数到分类概率

现在可以把逻辑回归最核心的预测过程写出来：

第一步，先算线性分数：

$$
z = w^T x + b
$$

第二步，再用 sigmoid 把它变成概率：

$$
p = \sigma(z) = \frac{1}{1+e^{-z}}
$$

并把这个概率解释成：

$$
p = P(y=1 \mid x)
$$

于是也就有：

$$
P(y=0 \mid x)=1-p
$$

这比线性回归多出来的一步，就是逻辑回归的核心变化。

你可以把它简单记成：

线性回归：

$$
\hat{y} = w^T x + b
$$

逻辑回归：

$$
\hat{p} = \sigma(w^T x + b)
$$

前者输出的是连续值，后者输出的是概率。

## 11. 阈值与决策边界

逻辑回归先输出概率，但最终分类时通常还要给出一个类别标签。

最常见的规则是使用 0.5 作为阈值：

- 如果 $P(y=1 \mid x) \ge 0.5$，预测为 1
- 如果 $P(y=1 \mid x) < 0.5$，预测为 0

由于：

$$
\sigma(0)=0.5
$$

所以分界点满足：

$$
w^T x + b = 0
$$

为了更清楚地理解这件事，我们把逻辑回归拆成两步：

第一步，先算线性分数：

$$
z = w^T x + b
$$

第二步，再把这个分数通过 sigmoid 映射成概率：

$$
p = \sigma(z)
$$

也就是说：

- 线性部分负责切分
- sigmoid 负责概率化

由于 sigmoid 是单调递增函数，所以它不会改变大小关系。也就是说：

$$
z_1 > z_2 \quad \Longrightarrow \quad \sigma(z_1) > \sigma(z_2)
$$

因此，用阈值 $0.5$ 做分类时：

$$
p \ge 0.5
$$

等价于：

$$
\sigma(z) \ge 0.5
$$

又因为：

$$
\sigma(0)=0.5
$$

所以这又等价于：

$$
z \ge 0
$$

也就是：

$$
w^T x + b \ge 0
$$

同理：

$$
p < 0.5
$$

等价于：

$$
w^T x + b < 0
$$

所以真正决定分类的是线性部分 $w^T x + b$ 的正负号，而不是 sigmoid 曲线本身的弯曲形状。

这说明一个很重要的事实：虽然逻辑回归输出的是非线性的 sigmoid 概率，但它的决策边界仍然是线性的。

如果在二维平面中，设

$$
x = \begin{bmatrix}x_1 \\ x_2\end{bmatrix},
\qquad
w = \begin{bmatrix}w_1 \\ w_2\end{bmatrix}
$$

那么边界方程就是：

$$
w_1 x_1 + w_2 x_2 + b = 0
$$

把它整理一下，就得到：

$$
x_2 = -\frac{w_1}{w_2}x_1 - \frac{b}{w_2}
$$

这就是一条直线。在更高维空间里，它对应的是一个超平面。

所以你可以把这件事记成一句话：

线性部分负责切分，sigmoid 负责概率化。

## 12. 为什么这会引出交叉熵

有了概率模型以后，我们就会自然地问：如果模型给出了一个概率 $p$，怎样衡量这个概率预测得好不好？

在线性回归里，我们常用平方误差：

$$
(y-\hat{y})^2
$$

但在逻辑回归中，预测对象已经变成了概率，而且标签服从伯努利分布，于是更自然的做法是从似然出发，得到对数损失，也就是后面会学到的二元交叉熵：

$$
-\left[y\log p + (1-y)\log(1-p)\right]
$$

这部分不用在本 Notebook 里完全展开，你现在只需要先建立这条联系：

- 二分类标签
- 伯努利分布
- 条件概率 $P(y=1\mid x)$
- sigmoid 输出概率
- 进一步引出交叉熵损失

这就是逻辑回归的数学主线。

## 14. 第一部分小结

可以把这份 Notebook 压缩成下面这条主线：

$$
y \in \{0,1\}
$$

$$
P(y=1\mid x)=p
$$

$$
\log\frac{p}{1-p}=w^Tx+b
$$

$$
p=\sigma(w^Tx+b)=\frac{1}{1+e^{-(w^Tx+b)}}
$$

这就是逻辑回归最核心的数学起点。

到这里为止，我们已经搭好了从线性模型走向概率模型的桥。

下一步就可以继续学习逻辑回归里最关键的损失函数：交叉熵。

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_values = np.array([-3, -1, 0, 1, 3])
p_values = sigmoid(z_values)

for z, p in zip(z_values, p_values):
    print(f"z = {z:>2}, sigmoid(z) = {p:.6f}")


## 15. 为什么逻辑回归通常不用 MSE，而用交叉熵

在线性回归里，我们常用均方误差：

$$
J_{\text{MSE}} = \frac{1}{m}\sum_{i=1}^{m}\left(y^{(i)}-\hat{y}^{(i)}\right)^2
$$

如果把逻辑回归中的预测概率记作 $p^{(i)}$，那么从形式上当然也可以写出：

$$
J = \frac{1}{m}\sum_{i=1}^{m}\left(y^{(i)}-p^{(i)}\right)^2
$$

也就是说，逻辑回归并不是“数学上完全不能写 MSE”。

但标准的逻辑回归通常不用它，而是使用交叉熵，原因主要有三点：

- 逻辑回归的标签是二分类标签，更自然地对应伯努利分布。
- 交叉熵可以从伯努利分布的似然函数直接推导出来。
- 交叉熵对“高置信度但预测错误”的情况惩罚更强，也更利于优化。

所以，交叉熵不是随便换一个损失函数，而是和逻辑回归的概率解释天然配套的。

## 16. 从伯努利分布到单个样本的似然

前面我们已经知道，对于单个样本，若模型输出：

$$
p = P(y=1 \mid x)
$$

那么也就有：

$$
P(y=0 \mid x)=1-p
$$

由于 $y\in\{0,1\}$，单个样本的条件概率可以统一写成：

$$
P(y\mid x)=p^y(1-p)^{1-y}
$$

这个式子本来是一个概率公式，但当我们把已经观察到的数据固定下来，再把它看成参数的函数时，它就叫作似然。

这里要区分两个很像但不完全一样的概念：

- 概率是“参数固定，看数据有多可能”
- 似然是“数据固定，看参数有多合理”

也就是说，当真实标签 $y$ 已经观察到以后，我们会问：哪一个 $p$ 更能解释这条已经出现的数据？

如果把似然函数写出来，可以记成：

$$
L(p\mid y)=p^y(1-p)^{1-y}
$$

你可以代入两种情况看得更直观一些。

当 $y=1$ 时：

$$
L(p\mid y=1)=p
$$

这说明：如果真实结果就是正类，那么模型给出的 $p$ 越大，这条数据在模型眼里就越“说得通”。

当 $y=0$ 时：

$$
L(p\mid y=0)=1-p
$$

这说明：如果真实结果是负类，那么模型给出的 $p$ 越小，就越合理。

所以我们想让似然尽可能大，本质上是因为：

$$
\text{我们希望模型把更高的概率分配给真实发生的结果。}
$$

这也可以理解成：

$$
\text{找到一组参数，使当前观察到的数据在这组参数下尽可能不意外。}
$$

在逻辑回归里，$p$ 不是一个随便给定的常数，而是由模型参数决定的：

$$
p = \sigma(w^T x + b)
$$

因此单个样本的似然也可以写成：

$$
L(w,b\mid x,y)=\left(\sigma(w^T x+b)\right)^y\left(1-\sigma(w^T x+b)\right)^{1-y}
$$

训练逻辑回归时，我们就是在寻找更好的 $w$ 和 $b$，让真实标签对应的似然尽可能大。

为了把“最大化似然”变成更方便优化的“最小化损失”，通常对它取对数，再加一个负号。于是得到负对数似然：

$$
L(y,p) = -\log\bigl(p^y(1-p)^{1-y}\bigr)
$$

再利用对数运算规则 $\log(ab)=\log a + \log b$，可得：

$$
L(y,p) = -\left[y\log p + (1-y)\log(1-p)\right]
$$

这就是二分类中单个样本的交叉熵损失。

## 17. 交叉熵在两种标签下的具体形式

交叉熵的统一公式是：

$$
L(y,p) = -\left[y\log p + (1-y)\log(1-p)\right]
$$

它在两种标签情况下会自动化简成非常直观的形式。

当 $y=1$ 时：

$$
L(1,p) = -\log p
$$

这表示：如果真实标签是 1，那么模型给正类的概率 $p$ 越大，损失越小。

当 $y=0$ 时：

$$
L(0,p) = -\log(1-p)
$$

这表示：如果真实标签是 0，那么模型给正类的概率 $p$ 越小，损失越小。

所以交叉熵的本质就是：

- 对真实类别给高概率，损失就小。
- 对真实类别给低概率，损失就大。

## 18. 所有样本的平均交叉熵

对于整个训练集，如果第 $i$ 个样本的预测概率记作：

$$
p^{(i)} = P(y^{(i)}=1\mid x^{(i)}) = \sigma\bigl(w^T x^{(i)} + b\bigr)
$$

那么整个数据集上的平均交叉熵损失写成：

$$
J(w,b) = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log p^{(i)} + (1-y^{(i)})\log(1-p^{(i)})\right]
$$

这就是逻辑回归最常见的目标函数。

训练逻辑回归，本质上就是在寻找参数 $w$ 和 $b$，使得上面的损失函数尽可能小。

## 19. 交叉熵为什么合理

交叉熵之所以合理，是因为它非常符合我们对“好预测”和“坏预测”的直觉。

### 19.1 当真实标签为 1 时

此时损失为：

$$
L(1,p) = -\log p
$$

如果 $p$ 很接近 1，那么 $\log p$ 接近 0，因此损失接近 0。

如果 $p$ 很接近 0，那么 $\log p$ 会变成一个很大的负数，因此损失会变得很大。

例如：

$$
-\log(0.9) \approx 0.105
$$

$$
-\log(0.1) \approx 2.303
$$

也就是说，如果真实标签是 1，而你只给出 0.1 的概率，那么惩罚会非常明显。

### 19.2 当真实标签为 0 时

此时损失为：

$$
L(0,p) = -\log(1-p)
$$

如果 $p$ 很接近 0，那么损失接近 0；如果 $p$ 很接近 1，那么损失会非常大。

这说明交叉熵特别擅长惩罚“非常自信但判断错了”的预测。

## 20. 交叉熵与 MSE 的对比

如果在逻辑回归里使用 MSE：

$$
J_{\text{MSE}} = \frac{1}{m}\sum_{i=1}^{m}\left(y^{(i)}-p^{(i)}\right)^2
$$

它当然也会衡量预测概率和真实标签之间的差距，但它并不是从伯努利分布自然推出的。

相比之下，交叉熵：

$$
J_{\text{CE}} = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log p^{(i)} + (1-y^{(i)})\log(1-p^{(i)})ight]
$$

与逻辑回归的概率建模完全对应。

因此可以把它们的差别简单记成：

- MSE 更像是在比较“数值差了多少”。
- 交叉熵更像是在比较“你给真实类别的概率到底够不够高”。

对于逻辑回归来说，后一种衡量方式更自然，也更符合模型的目标。

## 21. 本节小结

交叉熵这一节可以压缩成下面这条主线：

$$
P(y\mid x)=p^y(1-p)^{1-y}
$$

$$
L(y,p) = -\log P(y\mid x)
$$

$$
L(y,p) = -\left[y\log p + (1-y)\log(1-p)\right]
$$

$$
J(w,b) = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log p^{(i)} + (1-y^{(i)})\log(1-p^{(i)})ight]
$$

这就是逻辑回归中最经典的损失函数。

下一步继续往下学时，我们就可以进入：

- 交叉熵对参数的梯度
- 为什么梯度会化简得很漂亮
- 梯度下降如何训练逻辑回归

In [ ]:
import numpy as np

def binary_cross_entropy(y, p):
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

examples = [
    (1, 0.9),
    (1, 0.6),
    (1, 0.1),
    (0, 0.1),
    (0, 0.4),
    (0, 0.9),
]

for y, p in examples:
    loss = binary_cross_entropy(y, p)
    print(f"y={y}, p={p:.1f}, loss={loss:.6f}")
